# VGG16 + Random Forest — Augmented Dataset
Workflow: Nạp dữ liệu → Trích đặc trưng VGG16 → Huấn luyện RF → Đánh giá kết quả → Xuất model

## 1. Cài đặt & Nhập thư viện

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.utils import image_dataset_from_directory
from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.layers import Input, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, PredefinedSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd
import joblib
import time
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)
import joblib

print(f'TensorFlow version : {tf.__version__}')
print(f'GPU available      : {len(tf.config.list_physical_devices("GPU")) > 0}')

2026-04-27 10:09:24.429682: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777284564.609698      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777284564.661877      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777284565.074280      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777284565.074327      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777284565.074329      55 computation_placer.cc:177] computation placer alr

TensorFlow version : 2.19.0
GPU available      : True


## 2. Cấu hình đường dẫn (Kaggle)

In [2]:
# ── Kaggle: dataset được mount sẵn tại /kaggle/input/ ──
DATASET_ROOT = '/kaggle/input/datasets/trnnguynlmhuy/original-dataset'   # <-- thay bằng tên dataset Kaggle của bạn

DATA_SPLITS = {
    'train' : os.path.join(DATASET_ROOT, 'train'),
    'val'   : os.path.join(DATASET_ROOT, 'val'),
    'test'  : os.path.join(DATASET_ROOT, 'test'),
}

OUTPUT_DIR = '/kaggle/working/saved_models'
os.makedirs(OUTPUT_DIR, exist_ok=True)

for split, path in DATA_SPLITS.items():
    exists = os.path.isdir(path)
    print(f'  [{split}] {path}  →  {"OK" if exists else "NOT FOUND"}')

  [train] /kaggle/input/datasets/trnnguynlmhuy/original-dataset/train  →  OK
  [val] /kaggle/input/datasets/trnnguynlmhuy/original-dataset/val  →  OK
  [test] /kaggle/input/datasets/trnnguynlmhuy/original-dataset/test  →  OK


## 3. Nạp Dataset

In [3]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
SEED       = 2024

def load_split(directory, shuffle=False):
    """Trả về tf.data.Dataset từ thư mục ảnh có cấu trúc class-subfolder."""
    ds = image_dataset_from_directory(
        directory,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode='int',
        shuffle=shuffle,
        seed=SEED,
    )
    return ds

train_ds = load_split(DATA_SPLITS['train'], shuffle=False)
val_ds   = load_split(DATA_SPLITS['val'],   shuffle=False)
test_ds  = load_split(DATA_SPLITS['test'],  shuffle=False)

CLASS_NAMES  = train_ds.class_names
NUM_CLASSES  = len(CLASS_NAMES)
print(f'Số lớp phân loại : {NUM_CLASSES}')
print(f'Tên lớp          : {CLASS_NAMES[:5]} ...')

Found 20080 files belonging to 150 classes.


I0000 00:00:1777284865.500812      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1777284865.506974      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 4248 files belonging to 150 classes.
Found 4468 files belonging to 150 classes.
Số lớp phân loại : 150
Tên lớp          : ['ABBOTTS BOOBY', 'ABYSSINIAN GROUND HORNBILL', 'AFRICAN PIED HORNBILL', 'AFRICAN PYGMY GOOSE', 'ALPINE CHOUGH'] ...


## 4. Xây dựng Bộ Trích Đặc Trưng VGG16

In [4]:
def build_vgg16_extractor(img_shape=(224, 224, 3)):
    """
    VGG16 (ImageNet, không top) + GlobalAveragePooling2D.
    Toàn bộ trọng số bị đóng băng → chỉ dùng để trích đặc trưng.
    Output: vector 512 chiều mỗi ảnh.
    """
    inp = Input(shape=img_shape)
    base = VGG16(
        include_top=False,
        weights='imagenet',
        input_tensor=inp
    )
    base.trainable = False          # đóng băng hoàn toàn
    gap = GlobalAveragePooling2D()(base.output)
    extractor = Model(inputs=inp, outputs=gap, name='vgg16_gap_extractor')
    return extractor

feature_extractor = build_vgg16_extractor()
print(f'Feature vector dim   : {feature_extractor.output_shape}')   # (None, 512)
print(f'Trainable parameters : {feature_extractor.count_params()} (phải = 0 params trainable)')
feature_extractor.summary()

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Feature vector dim   : (None, 512)
Trainable parameters : 14714688 (phải = 0 params trainable)


Model: "vgg16_gap_extractor"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 224, 224, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 224, 224, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 112, 112, 128)  │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 112, 112, 128)  │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 56, 56, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 56, 56, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 28, 28, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 28, 28, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 28, 28, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 14, 14, 512)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 14, 14, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 7, 7, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 14,714,688 (56.13 MB)

## 5. Trích Xuất Đặc Trưng từ Toàn Bộ Tập Dữ Liệu

In [5]:
def extract_features_from_dataset(extractor, dataset, split_name=''):
    """
    Duyệt qua dataset → áp dụng VGG16 preprocessing → trích xuất đặc trưng.
    Trả về:
    - X: vector đặc trưng
    - y: nhãn
    """
    print(f'Trích đặc trưng — tập [{split_name}] ...')

    all_feats, all_labels = [], []

    for batch_imgs, batch_lbls in dataset:
        preprocessed = preprocess_input(tf.cast(batch_imgs, tf.float32))
        feats = extractor(preprocessed, training=False).numpy()

        all_feats.append(feats)
        all_labels.append(batch_lbls.numpy())

    X = np.concatenate(all_feats, axis=0)
    y = np.concatenate(all_labels, axis=0)

    print(f'  features shape: {X.shape} | labels shape: {y.shape}')

    return X, y


# Trích xuất đặc trưng cho từng tập
X_train, y_train = extract_features_from_dataset(feature_extractor, train_ds, 'train')
X_val,   y_val   = extract_features_from_dataset(feature_extractor, val_ds,   'val')
X_test,  y_test  = extract_features_from_dataset(feature_extractor, test_ds,  'test')


# ============================================================
# Gộp train + val để dùng cho PredefinedSplit
# ============================================================

X_combined = np.concatenate([X_train, X_val], axis=0)
y_combined = np.concatenate([y_train, y_val], axis=0)

# -1 nghĩa là phần này chỉ dùng để train
#  0 nghĩa là phần này dùng làm validation fold
test_fold = np.concatenate([
    np.full(len(y_train), -1),
    np.full(len(y_val), 0)
])

ps = PredefinedSplit(test_fold=test_fold)


# ============================================================
# Gộp train + val để sau này train model cuối cùng
# ============================================================

X_tv = X_combined
y_tv = y_combined


print(f'\nTrain set      : {X_train.shape}')
print(f'Validation set : {X_val.shape}')
print(f'Train + Val    : {X_tv.shape}')
print(f'Test set       : {X_test.shape}')

print(f'\nPredefinedSplit:')
print(f'Số mẫu train trong GridSearch : {(test_fold == -1).sum()}')
print(f'Số mẫu val trong GridSearch   : {(test_fold == 0).sum()}')


# Kiểm tra phân phối đặc trưng
print(f'\n--- Thống kê X_train+val ---')
print(f'Mean  : {X_tv.mean():.4f}')
print(f'Std   : {X_tv.std():.4f}')
print(f'Min   : {X_tv.min():.4f}')
print(f'Max   : {X_tv.max():.4f}')
print(f'% < 0 : {(X_tv < 0).mean()*100:.2f}%')

Trích đặc trưng — tập [train] ...


I0000 00:00:1777284978.954196      55 cuda_dnn.cc:529] Loaded cuDNN version 91002


  features shape: (20080, 512) | labels shape: (20080,)
Trích đặc trưng — tập [val] ...
  features shape: (4248, 512) | labels shape: (4248,)
Trích đặc trưng — tập [test] ...
  features shape: (4468, 512) | labels shape: (4468,)

Train set      : (20080, 512)
Validation set : (4248, 512)
Train + Val    : (24328, 512)
Test set       : (4468, 512)

PredefinedSplit:
Số mẫu train trong GridSearch : 20080
Số mẫu val trong GridSearch   : 4248

--- Thống kê X_train+val ---
Mean  : 3.3637
Std   : 5.4659
Min   : 0.0000
Max   : 160.2202
% < 0 : 0.00%


# Đo feature extraction time của CPU và GPU 

In [6]:
def measure_extraction_time_device(extractor, dataset, device_name, split_name='test', num_batches=10):
    """
    Đo Feature Extraction Time trên CPU hoặc GPU.
    Chỉ đo:
    ảnh -> preprocess -> VGG16 -> vector đặc trưng
    """

    print(f"\nĐo Feature Extraction Time trên {device_name} - tập [{split_name}] ...")

    total_time = 0.0
    total_images = 0

    with tf.device(device_name):
        # Warm-up
        for batch_imgs, _ in dataset.take(1):
            batch_imgs = tf.cast(batch_imgs, tf.float32)
            batch_imgs = preprocess_input(batch_imgs)
            feats = extractor(batch_imgs, training=False)
            _ = feats.numpy()   # ép TensorFlow chạy xong thật sự

        # Đo thời gian
        for batch_imgs, _ in dataset.take(num_batches):
            batch_size = batch_imgs.shape[0]

            start = time.perf_counter()

            batch_imgs = tf.cast(batch_imgs, tf.float32)
            batch_imgs = preprocess_input(batch_imgs)
            feats = extractor(batch_imgs, training=False)
            _ = feats.numpy()   # ép đồng bộ để đo đúng thời gian

            end = time.perf_counter()

            total_time += end - start
            total_images += batch_size

    time_per_image = total_time / total_images

    print(f"Tổng số ảnh đo       : {total_images}")
    print(f"Tổng thời gian       : {total_time:.4f} s")
    print(f"Extraction Time      : {time_per_image * 1000:.4f} ms/image")

    return time_per_image


# Kiểm tra máy có GPU không
gpus = tf.config.list_physical_devices('GPU')
print("GPU available:", gpus)

# Đo CPU
cpu_ext_time = measure_extraction_time_device(
    extractor=feature_extractor,
    dataset=test_ds,
    device_name='/CPU:0',
    split_name='test',
    num_batches=10
)

# Đo GPU nếu có
if len(gpus) > 0:
    gpu_ext_time = measure_extraction_time_device(
        extractor=feature_extractor,
        dataset=test_ds,
        device_name='/GPU:0',
        split_name='test',
        num_batches=10
    )

    print("\n===== Feature Extraction Time CPU/GPU =====")
    print(f"CPU: {cpu_ext_time * 1000:.4f} ms/image")
    print(f"GPU: {gpu_ext_time * 1000:.4f} ms/image")
else:
    gpu_ext_time = None

    print("\n===== Feature Extraction Time CPU/GPU =====")
    print(f"CPU: {cpu_ext_time * 1000:.4f} ms/image")
    print("GPU: Không có GPU hoặc GPU chưa được bật")

GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

Đo Feature Extraction Time trên /CPU:0 - tập [test] ...
Tổng số ảnh đo       : 320
Tổng thời gian       : 61.7518 s
Extraction Time      : 192.9742 ms/image

Đo Feature Extraction Time trên /GPU:0 - tập [test] ...
Tổng số ảnh đo       : 320
Tổng thời gian       : 1.9210 s
Extraction Time      : 6.0031 ms/image

===== Feature Extraction Time CPU/GPU =====
CPU: 192.9742 ms/image
GPU: 6.0031 ms/image


## 6. Huấn Luyện Random Forest bằng gridsearch tìm tham số tốt nhất

In [7]:
# ============================================================
# Grid Search Random Forest trên vector đặc trưng VGG16
# ============================================================

param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20],
    "min_samples_split": [2, 10],
    "min_samples_leaf": [1, 5],
    "max_features": ["sqrt"],
    "bootstrap": [True]
}

rf = RandomForestClassifier(
    n_jobs=-1,
    random_state=42
)

grid = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=ps,                 # dùng validation folder đã chia sẵn
    scoring="accuracy",
    verbose=3,
    n_jobs=-1,
)

print("Bắt đầu chạy Grid Search...")
grid.fit(X_tv, y_tv)
print("Hoàn thành Grid Search!")

print("\nBest params:")
print(grid.best_params_)

print(f"\nBest CV accuracy: {grid.best_score_:.4f}")

Bắt đầu chạy Grid Search...
Fitting 1 folds for each of 16 candidates, totalling 16 fits
Hoàn thành Grid Search!

Best params:
{'bootstrap': True, 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 5, 'min_samples_split': 2, 'n_estimators': 200}

Best CV accuracy: 0.8451
[CV 1/1] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=200;, score=0.767 total time= 4.9min
[CV 1/1] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=10, n_estimators=200;, score=0.770 total time= 4.7min
[CV 1/1] END bootstrap=True, max_depth=20, max_features=sqrt, min_samples_leaf=5, min_samples_split=2, n_estimators=100;, score=0.821 total time= 3.5min
[CV 1/1] END bootstrap=True, max_depth=20, max_features=sqrt, min_samples_leaf=5, min_samples_split=10, n_estimators=100;, score=0.821 total time= 3.4min
[CV 1/1] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, 

## 7. Train random forest với bộ tham số tốt nhất tìm được sau khi chạy gridsearch

In [8]:
# ============================================================
# Train Random Forest cuối cùng với best params
# ============================================================

best_params = grid.best_params_

best_rf = RandomForestClassifier(
    **best_params,
    n_jobs=-1,
    random_state=42
)

print("Bắt đầu train Random Forest với best params...")
best_rf.fit(X_tv, y_tv)
print("Hoàn thành train model cuối!")

Bắt đầu train Random Forest với best params...
[CV 1/1] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=1, min_samples_split=10, n_estimators=100;, score=0.737 total time= 2.4min
[CV 1/1] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=2, n_estimators=100;, score=0.738 total time= 2.4min
[CV 1/1] END bootstrap=True, max_depth=10, max_features=sqrt, min_samples_leaf=5, min_samples_split=10, n_estimators=100;, score=0.738 total time= 2.4min
[CV 1/1] END bootstrap=True, max_depth=20, max_features=sqrt, min_samples_leaf=1, min_samples_split=2, n_estimators=200;, score=0.842 total time= 7.8min
[CV 1/1] END bootstrap=True, max_depth=20, max_features=sqrt, min_samples_leaf=5, min_samples_split=10, n_estimators=200;, score=0.845 total time= 3.2min
Hoàn thành train model cuối!


In [9]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =========================
# Predict
# =========================
y_train_pred = best_rf.predict(X_tv)
y_test_pred = best_rf.predict(X_test)

# =========================
# Accuracy
# =========================
train_acc = accuracy_score(y_tv, y_train_pred)
test_acc = accuracy_score(y_test, y_test_pred)

print(f"Train Accuracy: {train_acc:.4f}")
print(f"Test Accuracy:  {test_acc:.4f}")

# =========================
# Classification Report
# =========================
print("\n=== Train Classification Report ===")
print(classification_report(y_tv, y_train_pred))

print("\n=== Test Classification Report ===")
print(classification_report(y_test, y_test_pred))

# =========================
# Confusion Matrix
# =========================
print("\n=== Confusion Matrix (Test) ===")
print(confusion_matrix(y_test, y_test_pred))

Train Accuracy: 0.9986
Test Accuracy:  0.8487

=== Train Classification Report ===
              precision    recall  f1-score   support

           0       1.00      0.99      0.99       156
           1       1.00      1.00      1.00       155
           2       1.00      1.00      1.00       162
           3       1.00      1.00      1.00       155
           4       1.00      1.00      1.00       144
           5       1.00      1.00      1.00       155
           6       1.00      0.99      1.00       148
           7       1.00      1.00      1.00       163
           8       0.99      0.99      0.99       155
           9       1.00      1.00      1.00       162
          10       0.99      1.00      0.99       163
          11       1.00      1.00      1.00       187
          12       1.00      1.00      1.00       165
          13       1.00      1.00      1.00       162
          14       1.00      1.00      1.00       165
          15       1.00      0.99      1.00       14

In [10]:
report = classification_report(y_test, y_test_pred, output_dict=True)

print("Precision (macro):", report["macro avg"]["precision"])
print("Recall (macro):   ", report["macro avg"]["recall"])
print("F1-score (macro): ", report["macro avg"]["f1-score"])

Precision (macro): 0.8598659507936611
Recall (macro):    0.8482850439568347
F1-score (macro):  0.8479454481378343


# Tính inference CPU

In [11]:
def measure_inference_time_cpu(feature_extractor, rf_model, dataset, num_batches=10):
    """
    Đo Inference Time CPU trung bình trên 1 ảnh.
    Bao gồm:
    - preprocess ảnh
    - VGG16 trích xuất đặc trưng
    - Random Forest dự đoán
    """

    total_time = 0.0
    total_images = 0

    # Warm-up để lần chạy đầu không bị lệch thời gian
    for batch_imgs, _ in dataset.take(1):
        batch_imgs = preprocess_input(tf.cast(batch_imgs, tf.float32))
        feats = feature_extractor(batch_imgs, training=False).numpy()
        _ = rf_model.predict(feats)

    # Đo thời gian inference
    for batch_imgs, _ in dataset.take(num_batches):
        batch_size = batch_imgs.shape[0]

        start = time.perf_counter()

        batch_imgs = preprocess_input(tf.cast(batch_imgs, tf.float32))
        feats = feature_extractor(batch_imgs, training=False).numpy()
        _ = rf_model.predict(feats)

        end = time.perf_counter()

        total_time += end - start
        total_images += batch_size

    avg_time = total_time / total_images

    return avg_time


inference_time_cpu = measure_inference_time_cpu(
    feature_extractor=feature_extractor,
    rf_model=best_rf,
    dataset=test_ds,
    num_batches=10
)

print(f"Inference Time CPU: {inference_time_cpu:.6f} giây/ảnh")
print(f"Inference Time CPU: {inference_time_cpu * 1000:.3f} ms/ảnh")

Inference Time CPU: 0.007967 giây/ảnh
Inference Time CPU: 7.967 ms/ảnh


## 8. Lưu Model

In [12]:
rf_path = os.path.join(OUTPUT_DIR, 'vgg16_rf_ori.joblib')

joblib.dump(best_rf, rf_path)

rf_size_mb = os.path.getsize(rf_path) / (1024 * 1024)

print(f"Random Forest model path: {rf_path}")
print(f"Random Forest model size: {rf_size_mb:.2f} MB")

Random Forest model path: /kaggle/working/saved_models/vgg16_rf_ori.joblib
Random Forest model size: 907.24 MB
